# Study 927 — Dutch Auction 🔨

**When a board pays a premium to buy back its own stock in a single fixed auction, does the
tape reward whoever buys in behind it?**

A *modified Dutch auction* self-tender is the loudest repurchase there is. The company posts
a price range, invites holders to name their price, and buys a large block at one clearing
price inside a twenty-business-day window — usually starting at a **premium** to the market.
The folklore reads it as insiders declaring the stock cheap, and expects the stock to keep
out-performing afterwards.

We test it on **145 self-tenders** across **109 issuers**,
2010-06-18 → 2025-11-21, harvested by a single EDGAR full-text-search query over form
**SC TO-I** (so every event is checkable by accession number, not remembered). Abnormal
return = issuer **minus SPY**, daily **total-return** closes, one execution lag.

*Real-tape numbers below are the frozen headline (`docs/results.md`, Fingerprint
`c39ea6c65ccb`); the live cells run the offline **synthetic** control and are labelled as such.
As-of 2026-06-30.*


## 1. What actually happens on the day

Forget theory for a second. Here is what the tape does across 145 of these offers — the stock's move *minus* the S&P 500's, so a market rally cannot be mistaken for good news about the company.

In [1]:
R = dict(ar0=4.72, ar0_t=7.93, ar0_hit=83, win=0.12, win_t=0.17, m6=2.76, m6_med=-2.36, m6_t=0.71)
print('Announcement day     : %+.2f%% vs the market   (t = %+.2f, positive in %d%% of cases)'
      % (R['ar0'], R['ar0_t'], R['ar0_hit']))
print('The tender window    : %+.2f%% vs the market   (t = %+.2f)'
      % (R['win'], R['win_t']))
print('Six months after     : %+.2f%% on average, but %+.2f%% for the TYPICAL one (t = %+.2f)'
      % (R['m6'], R['m6_med'], R['m6_t']))

Announcement day     : +4.72% vs the market   (t = +7.93, positive in 83% of cases)
The tender window    : +0.12% vs the market   (t = +0.17)
Six months after     : +2.76% on average, but -2.36% for the TYPICAL one (t = +0.71)


## 2. The whole story is one day wide

On the day the tender documents hit EDGAR, the stock jumps **4.72%** more than the market — and it does so **83% of the time**. That is not a subtle statistical whisper; it is one of the cleanest event effects on this desk.

And then it stops. Buy at the close of the *next* session — the first moment you could actually act on the filing — and the tender window pays **+0.12%** against the market. Six months after the offer expires, the *average* is **+2.76%** but the *typical* company is **-2.36%** — the average is being carried by a handful of names that went on to do well for reasons that have nothing to do with the buyback.

> 🔬 **For the quants.** Announcement *t* = +7.93 with a leave-one-out range of [+7.81, +8.86]; the tradable window *t* = +0.17 and the six-month *t* = +0.71, both with bootstrap CIs straddling zero.

## 3. How do we know the jump is not just noise?

Three ways.

**It is date-locked.** Move the assumed announcement day by one session and the effect vanishes: +0.64% at day −1, **+4.72% at day 0**, +0.43% at day +1, +0.00% at day +5. Whatever is happening happens *exactly* when the filing lands.

**A placebo cannot fake it.** Re-run the same 145 companies on randomly chosen dates on their own price histories, 2,000 times: the random version averages **+0.01%**. The real one is **+4.72%** — *p* = 0.0005.

**It survives the split.** +3.10% in 2010–2017 and +5.97% in 2018–2025. Both halves, same sign, both convincing.

## 4. So why can't you trade it?

Because you are not in the room. The offer *becomes* public on the day it moves — there is no window in which you know about the auction and the price has not already adjusted. Everything after that is flat.

We built the obvious sleeve anyway: buy each issuer one session after the filing, hold to expiry, short the S&P against it. Over 4,046 sessions that portfolio compounds to **-69.2%** after costs, with a Sharpe of **-0.302**. A long-only version that just holds the event names and sits in T-bills otherwise earns an excess-of-cash Sharpe of **+0.111** against SPY's **+0.809** — you would have been better off owning the index and never reading a filing.

> 🔬 **For the quants.** HAC *t* on the daily *mean* difference between the net event basket and SPY, both excess of BIL: **-2.00** — right on the two-sigma line. The wide number is the Sharpe gap; the mean-difference test only just calls it at conventional size, and we say so rather than rounding it up.

## 5. Is the effect at least bigger where you *could* trade?

No — it is the other way round, as it usually is. Restrict to the 60 events on names trading at least $10m a day and the announcement pop shrinks from 4.72% to **3.43%**. The biggest jumps belong to the smallest, thinnest issuers — where a tender for 10% of the float is genuinely transformative and where your own order would move the price.

## 6. A live check that the machinery works (offline synthetic)

**This cell does not touch the real tape.** It builds an artificial world where we *plant* both an announcement pop and a six-month drift, then a second world with neither, and checks the same code finds the first and stays quiet on the second. If the harness could not see a planted drift, the flat real-tape drift would prove nothing.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from dutch_auction import data, strategy as st

pl_px, pl_ev, truth = data.synthetic_panel(n_events=60, n_days=900,
                                           signal_strength=1.0, seed=927)
nl_px, nl_ev, _ = data.synthetic_panel(n_events=60, n_days=900,
                                       signal_strength=0.0, seed=927)
pl = st.synthetic_detect(pl_px, pl_ev, market='MKT')
nl = st.synthetic_detect(nl_px, nl_ev, market='MKT')
print('SYNTHETIC (not the real tape)')
print('  planted world: jump planted %+.0f bps -> recovered %+.0f bps (t=%+.2f)'
      % (truth['planted_jump']*1e4, pl['ar0_bps'], pl['t_ar0']))
print('  planted world: 6m drift planted %+.0f bps of log return, which is %+.0f bps'
      % (truth['planted_drift_6m_log']*1e4, truth['expected_simple_drift_6m']*1e4))
print('                 of expected buy-and-hold return -> recovered %+.0f bps (t=%+.2f)'
      % (pl['m6_bps'], pl['t_m6']))
print('  null world   : day-0 %+.0f bps (t=%+.2f)  <- must be ~0'
      % (nl['ar0_bps'], nl['t_ar0']))

SYNTHETIC (not the real tape)
  planted world: jump planted +470 bps -> recovered +470 bps (t=+17.28)
  planted world: 6m drift planted +300 bps of log return, which is +607 bps
                 of expected buy-and-hold return -> recovered +1195 bps (t=+3.28)
  null world   : day-0 -11 bps (t=-0.44)  <- must be ~0


## Verdict

- **Signal — Mixed.** Two claims, two answers. The announcement repricing is **real and large** (+4.72%, *t* = +7.93, 83% of 145 events, placebo *p* = 0.0005, date-locked, present in both eras). The "it marks the bottom" claim is **absent**: the tender window pays +0.12% (*t* = +0.17) and six months later the median company is -2.36% behind the market.
- **Tradability — Mirage.** The one day that pays is the one day you cannot be positioned for. Everything after it is zero gross and negative net: -0.28% per event at 10 bps, a -0.302 Sharpe for the long/short sleeve, and -0.698 Sharpe *behind* simply owning SPY.